# Use SparqlHelper with WikiPathways

Load published queries, give them names, find the classes of pathway IRIs, and save your collection.

We use three [WikiPathways examples](https://github.com/wikipathways/sparql-examples/tree/main/examples): dataset metadata (001), organisms (006), and mouse pathways (009). The query files are pinned to a repository revision; endpoint results can change.

Use a Python kernel with this checkout of `rdfsolve`, `pandas`, and `ipykernel` installed. Run the cells from top to bottom. This notebook makes a few sequential requests; it does not mine the endpoint.

## 1. Start a session

A short line reports whether each request used an HTTP fallback. Page-size reductions and retries also appear if needed. Query text and page-by-page progress stay hidden. We do not force a timeout just to demonstrate recovery.

In [1]:
import logging

import pandas as pd
import requests
from rdflib import Graph

from rdfsolve.sparql_helper import SparqlHelper

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s", force=True)
logging.getLogger("rdfsolve.sparql_helper").setLevel(logging.INFO)

helper = SparqlHelper(
    "https://sparql.wikipathways.org/sparql",
    timeout=30, max_retries=2, inter_request_delay=1,
)

## 2. Load published queries

Loading does not run a query. Here we download three small Turtle files, then give each query a name. Their original RDF descriptions remain in the collection.

In [2]:
base = "https://raw.githubusercontent.com/wikipathways/sparql-examples/3032382efd9c60c3ded84786b1bff099645f1e4b/examples/WikiPathways"
examples = {"001": "dataset metadata", "006": "organisms", "009": "mouse pathways"}

for number, name in examples.items():
    response = requests.get(f"{base}/{number}.ttl", timeout=20)
    response.raise_for_status()
    names = helper.load_shacl(Graph().parse(data=response.text, format="turtle"))
    helper.queries.rename(names[0], name)

list(helper.queries.queries)

['dataset metadata', 'organisms', 'mouse pathways']

## 3. Read and run a query

Inspect a saved query before running it. This one lists organisms represented in WikiPathways. We display the first eight rows.

In [3]:
print(helper.queries.queries["organisms"].query)

PREFIX wp: <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?organism (str(?label) as ?name)
WHERE {
    ?concept wp:organism ?organism ;
      wp:organismName ?label .
}


In [4]:
results = helper.run_query("organisms")
pd.json_normalize(results["results"]["bindings"]).filter(like=".value").head(8)

INFO: SELECT completed (no HTTP fallback)


,organism.value,name.value
0,http://purl.obolibrary.org/obo/NCBITaxon_9913,Bos taurus
1,http://purl.obolibrary.org/obo/NCBITaxon_9615,Canis familiaris
2,http://purl.obolibrary.org/obo/NCBITaxon_10090,Mus musculus
3,http://purl.obolibrary.org/obo/NCBITaxon_10116,Rattus norvegicus
4,http://purl.obolibrary.org/obo/NCBITaxon_7955,Danio rerio
5,http://purl.obolibrary.org/obo/NCBITaxon_6239,Caenorhabditis elegans
6,http://purl.obolibrary.org/obo/NCBITaxon_9606,Homo sapiens
7,http://purl.obolibrary.org/obo/NCBITaxon_4932,Saccharomyces cerevisiae


## 4. Retrieve a few mouse pathways

Keep the published query unchanged and save a second version with a five-row limit. The original already has an order, so we only add the limit.

In [5]:
query = helper.queries.queries["mouse pathways"].query
helper.add_query("five mouse pathways", query + "\nLIMIT 5")
pathways = helper.run_query("five mouse pathways")
pd.json_normalize(pathways["results"]["bindings"]).filter(like=".value")

INFO: SELECT completed (no HTTP fallback)


,wpIdentifier.value,pathway.value,page.value
0,https://identifiers.org/wikipathways/WP1,https://identifiers.org/wikipathways/WP1_r137182,http://www.wikipathways.org/instance/WP1_r137182
1,https://identifiers.org/wikipathways/WP10,https://identifiers.org/wikipathways/WP10_r139876,http://www.wikipathways.org/instance/WP10_r139876
2,https://identifiers.org/wikipathways/WP103,https://identifiers.org/wikipathways/WP103_r13...,http://www.wikipathways.org/instance/WP103_r13...
3,https://identifiers.org/wikipathways/WP108,https://identifiers.org/wikipathways/WP108_r14...,http://www.wikipathways.org/instance/WP108_r14...
4,https://identifiers.org/wikipathways/WP113,https://identifiers.org/wikipathways/WP113_r13...,http://www.wikipathways.org/instance/WP113_r13...


## 5. What classes do these IRIs have?

No need to write a new query. Pass the returned IRIs to the class lookup. It batches longer lists automatically.

These are `rdf:type` statements found in the data—not proof that each type has an OWL class declaration. An absent IRI means no type was returned in the queried scope. A failed request raises an error instead of looking empty.

In [6]:
iris = [row["pathway"]["value"] for row in pathways["results"]["bindings"]]
helper.find_classes_for_iris(iris)

INFO: SELECT completed (no HTTP fallback)


{'https://identifiers.org/wikipathways/WP103_r136920': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP108_r142196': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP113_r137233': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP10_r139876': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway'],
 'https://identifiers.org/wikipathways/WP1_r137182': ['http://www.w3.org/2004/02/skos/core#Collection',
  'http://vocabularies.wikipathways.org/wp#Pathway']}

Need to know **which named graph** supplied a type? Use the graph-aware lookup. It does not include the default graph.

In [7]:
helper.find_classes_for_iris_by_graph(iris[:1])

INFO: SELECT completed (no HTTP fallback)


{'https://identifiers.org/wikipathways/WP1_r137182': {'http://rdf.wikipathways.org/': ['http://www.w3.org/2004/02/skos/core#Collection',
   'http://vocabularies.wikipathways.org/wp#Pathway']}}

## 6. Use a SHACL path to retrieve a label

The published files contain queries, not property-shape paths. We add a small shape of our own: read either a title or a label. This describes what to retrieve; it does not require the data to have either property.

In [8]:
labels = Graph().parse(data="""
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix dc: <http://purl.org/dc/elements/1.1/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    <urn:pathway-label> sh:path [ sh:alternativePath (dc:title rdfs:label) ] .
""", format="turtle")
helper.load_shacl(labels)

query = helper.queries.path_query("urn:pathway-label", iris[0], limit=5)
helper.add_query("pathway label", query)
helper.run_query("pathway label")["results"]["bindings"]

INFO: SELECT completed (no HTTP fallback)


[{'value': {'type': 'literal', 'xml:lang': 'en', 'value': 'Statin pathway'}}]

The same method accepts sequence and inverse paths from a loaded SHACL document. It builds a query for one IRI; it does not hydrate Python objects. A result limit bounds the output, not the cost of following a path.

## 7. Save the collection and review the session

Turtle contains the saved queries and loaded shapes, not their results or run history. Use `load_shacl()` to reopen it in another helper. The file below is written beside the notebook when that is your working directory.

In [9]:
from pathlib import Path

output = Path("wikipathways_queries.ttl")
helper.export_queries_as_ttl(output)
print(output.resolve())

pd.DataFrame(helper.history)

/trinity/home/javier.millanacosta/rdfsolve/rdfsolve-2/notebooks/SparqlHelper/wikipathways_queries.ttl


,name,endpoint,started_at,elapsed_seconds,success,error
0,organisms,https://sparql.wikipathways.org/sparql,2026-09-08T11:01:19.424382+00:00,0.148798,True,
1,five mouse pathways,https://sparql.wikipathways.org/sparql,2026-09-08T11:01:19.590026+00:00,0.866863,True,
2,pathway label,https://sparql.wikipathways.org/sparql,2026-09-08T11:01:22.613644+00:00,0.843349,True,


History covers `run_query()` calls; the class lookups report through the log. Success means a request returned successfully, not that the endpoint supplied every possible result.

For a query that returns too much data, use `prepare_paginated_query()` and `select_chunked()` as shown in the README. Use a stable order; paging cannot make every expensive query cheap.

Close the HTTP session when finished. To repeat this notebook, start again at the setup cell.

In [10]:
helper.close()